# HPM Digital Twin V0.5：动态感知—防护闭环快速浏览

本 Notebook 默认读取已经生成的快速配置结果，不会自动执行耗时的 Monte Carlo 全流程。所有功率均相对白噪声方差归一化；不涉及真实源预算、射程或设备毁伤阈值。

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, SVG

ROOT = Path.cwd()
if not (ROOT / 'outputs_v05_closed_loop').exists():
    ROOT = ROOT.parent
OUT = ROOT / 'outputs_v05_closed_loop'
assert OUT.exists(), f'未找到输出目录: {OUT}'
print('项目根目录:', ROOT.resolve())
print('结果目录:', OUT.resolve())

## 1. 闭环机理图

In [ ]:
display(SVG(filename=str(OUT / '00_pcp_closed_loop_mechanism.svg')))

## 2. 代表序列关键指标

In [ ]:
summary = json.loads((OUT / 'results_summary.json').read_text(encoding='utf-8'))
rep = pd.DataFrame(summary['representative']['method_summary'])
cols = ['method', 'mean_output_sinr_db', 'p05_output_sinr_db', 'protection_availability', 'mean_worst_response_db', 'median_design_runtime_ms']
display(rep[cols].round(3))
print(f"PAWR更新点平均测向误差: {summary['representative']['mean_measurement_error_deg']:.3f}°")
print(f"感知与协方差提取中位耗时: {summary['representative']['median_perception_runtime_ms']:.2f} ms")

## 3. 双干扰轨迹与逐帧输出 SINR

In [ ]:
display(Image(filename=str(OUT / '01_dynamic_trajectories.png'), width=900))
display(Image(filename=str(OUT / '03_output_sinr_timeline.png'), width=900))

## 4. 关键帧二维接收响应对比

In [ ]:
display(Image(filename=str(OUT / '08_delayed_point_response_map.png'), width=760))
display(Image(filename=str(OUT / '09_pcp_response_map.png'), width=760))

## 5. 4帧滞后 Monte Carlo 结果与配对统计

In [ ]:
lat4 = pd.DataFrame(summary['stressed_latency_results'])
show_cols = ['method', 'n_trials', 'mean_output_sinr_db_mean', 'mean_output_sinr_db_ci_low', 'mean_output_sinr_db_ci_high', 'protection_availability_mean']
display(lat4[show_cols].round(3))
stats = pd.DataFrame(summary['paired_statistics']).T.reset_index(names='baseline')
display(stats.round(4))

## 6. 消融实验

In [ ]:
display(pd.DataFrame(summary['ablation_summary']).round(3))
display(Image(filename=str(OUT / '18_ablation.png'), width=900))

## 7. 重新运行

快速配置：

```bash
python run_closed_loop_v05.py
```

论文级配置（100次/点，尚未在本交付中执行）：

```bash
python run_closed_loop_v05.py --config configs/closed_loop_v05_paper.yaml --output outputs_v05_paper
```